<a href="https://colab.research.google.com/github/viniciusdev772/IA/blob/main/treino_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GemmaMicro — Treino PT-BR no Colab

**Ordem de execução:** rode as células de cima para baixo.

- GPU recomendada: T4 (gratuita) ou A100 (Colab Pro)
- Salva automaticamente no Google Drive
- Tempo estimado: ~2h (T4) / ~40min (A100)

## 1. Verifica GPU e instala dependências

In [5]:
import subprocess, sys

# Checa GPU
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
print('GPU:', result.stdout.strip() or 'NENHUMA — vá em Runtime > Change runtime type > T4 GPU')

# Instala dependências
!pip install -q torch tokenizers datasets trafilatura requests
print('Dependências instaladas.')

GPU: Tesla T4, 15360 MiB
Dependências instaladas.


## 2. Monta Google Drive (salva modelo aqui)

In [6]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/GemmaMicro'
os.makedirs(DRIVE_DIR, exist_ok=True)

# Testa escrita
_t = f'{DRIVE_DIR}/_test.txt'
open(_t, 'w').write('ok')
assert os.path.exists(_t)
os.remove(_t)
print(f'Drive OK → {DRIVE_DIR}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive OK → /content/drive/MyDrive/GemmaMicro


## 3. Clona o repositório

In [7]:
import os

REPO = '/content/IA'
if not os.path.exists(REPO):
    !git clone https://github.com/viniciusdev772/IA.git {REPO}
else:
    !git -C {REPO} pull

print('Repo pronto.')

remote: Enumerating objects: 5, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 3 (delta 2), reused 3 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (3/3), 671 bytes | 671.00 KiB/s, done.
From https://github.com/viniciusdev772/IA
   6f37f4b..d222f2e  main       -> origin/main
Updating 6f37f4b..d222f2e
Fast-forward
 treino_colab.ipynb | 245 ++++-------------------------------------------------
 1 file changed, 18 insertions(+), 227 deletions(-)
Repo pronto.


## 4. Coleta dados via crawler PT-BR

Crawla Wikipedia PT, G1, Agência Brasil, Folha, BBC Brasil, Nexo, etc.  
Sem downloads gigantes — coleta diretamente da web e salva parágrafos limpos.  
**Tempo estimado: 10-20 min** (2000 páginas, 16 threads paralelas).

In [ ]:
import sys, os, shutil
sys.path.insert(0, '/content/IA')

REPO      = '/content/IA'
DATA_DIR  = '/content/datasets'
DRIVE_DIR = '/content/drive/MyDrive/GemmaMicro'
CRAWL_OUT = f'{DATA_DIR}/dataset_crawl.txt'

os.makedirs(DATA_DIR, exist_ok=True)

# ── Reutiliza crawl do Drive se já existir ─────────────────────────────────
drive_crawl = f'{DRIVE_DIR}/dataset_crawl.txt'
if not os.path.exists(CRAWL_OUT) and os.path.exists(drive_crawl):
    shutil.copy(drive_crawl, CRAWL_OUT)
    print(f'Crawl reutilizado do Drive: {os.path.getsize(CRAWL_OUT)/1024/1024:.1f} MB')

# ── Roda crawler se necessário ─────────────────────────────────────────────
if not os.path.exists(CRAWL_OUT):
    print('Iniciando crawler PT-BR (3000 páginas)...')

    import importlib

    # exec_module executa o código do módulo (redefine OUTPUT/MAX_PAGES/WORKERS),
    # então as sobreescritas devem vir DEPOIS de exec_module
    spec = importlib.util.spec_from_file_location('crawl', f'{REPO}/crawl.py')
    crawl_mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(crawl_mod)

    # Sobrescreve após exec_module para não ser resetado
    crawl_mod.OUTPUT    = CRAWL_OUT
    crawl_mod.MAX_PAGES = 3000
    crawl_mod.WORKERS   = 16

    crawl_mod.crawl()

    if not os.path.exists(CRAWL_OUT):
        raise FileNotFoundError(f'Crawler terminou mas {CRAWL_OUT} não foi criado — verifique crawl.py')

    print(f'\nCrawl concluído: {os.path.getsize(CRAWL_OUT)/1024/1024:.1f} MB')

    # ── Limpa e deduplica via clean_crawl ─────────────────────────────────
    spec2 = importlib.util.spec_from_file_location('clean_crawl', f'{REPO}/clean_crawl.py')
    clean_mod = importlib.util.module_from_spec(spec2)
    spec2.loader.exec_module(clean_mod)
    clean_mod.INPUT  = CRAWL_OUT
    clean_mod.OUTPUT = CRAWL_OUT
    clean_mod.main()

    print(f'Dataset limpo: {os.path.getsize(CRAWL_OUT)/1024/1024:.1f} MB')

    # Salva cópia no Drive para reuso futuro
    shutil.copy(CRAWL_OUT, drive_crawl)
    print(f'Backup no Drive: {drive_crawl}')
else:
    print(f'dataset_crawl.txt já existe ({os.path.getsize(CRAWL_OUT)/1024/1024:.1f} MB), pulando crawler.')


Iniciando crawler PT-BR (3000 páginas)...
Iniciando crawl paralelo: 3000 páginas alvo | 16 workers



[0001/3000] 737 novos | total 729 | https://pt.wikipedia.org/wiki/Hist%C3%B3ria_do_Brasil
[0002/3000] 526 novos | total 1,249 | https://pt.wikipedia.org/wiki/Sociologia
[0003/3000] 656 novos | total 1,875 | https://pt.wikipedia.org/wiki/Brasil
[0004/3000] 197 novos | total 2,066 | https://pt.wikipedia.org/wiki/Intelig%C3%AAncia_artificial
[0005/3000] 131 novos | total 2,197 | https://pt.wikipedia.org/wiki/Astronomia
[0006/3000] 404 novos | total 2,581 | https://pt.wikipedia.org/wiki/F%C3%ADsica
[0007/3000] 429 novos | total 2,981 | https://pt.wikipedia.org/wiki/Ci%C3%AAncia


[0008/3000] 291 novos | total 3,272 | https://pt.wikipedia.org/wiki/Rede_neural_artificial
[0009/3000] 478 novos | total 3,746 | https://pt.wikipedia.org/wiki/Guerra_Fria
[0010/3000] 938 novos | total 4,646 | https://pt.wikipedia.org/wiki/Aquecimento_global
[0011/3000] 531 novos | total 5,168 | https://pt.wikipedia.org/wiki/Uni%C3%A3o_Europeia
[0012/3000]   4 novos | total 5,172 | https://agenciabrasil.ebc.com.br/cultura
[0013/3000]   6 novos | total 5,178 | https://agenciabrasil.ebc.com.br/educacao
[0014/3000]   7 novos | total 5,185 | https://agenciabrasil.ebc.com.br/
[0015/3000]   6 novos | total 5,191 | https://agenciabrasil.ebc.com.br/meio-ambiente
[0016/3000]   2 novos | total 5,193 | https://agenciabrasil.ebc.com.br/saude
[0017/3000]   9 novos | total 5,202 | https://agenciabrasil.ebc.com.br/direitos-humanos
[0018/3000]  13 novos | total 5,214 | https://brasilescola.uol.com.br/
[0019/3000]  15 novos | total 5,229 | https://www.infoescola.com/
[0020/3000]  87 novos | total 5,316 

[0058/3000]  16 novos | total 6,802 | https://aosfatos.org/
[0059/3000] 240 novos | total 7,042 | https://pt.wikipedia.org/wiki/Energia_nuclear
[0060/3000] 555 novos | total 7,566 | https://pt.wikipedia.org/wiki/Haiti
[0061/3000] 347 novos | total 7,900 | https://pt.wikipedia.org/wiki/Língua_portuguesa
[0062/3000]  97 novos | total 7,997 | https://pt.wikipedia.org/wiki/Caio_Prado_Júnior
[0063/3000]  40 novos | total 8,037 | https://pt.wikipedia.org/wiki/Guadalupe


[0064/3000]  21 novos | total 8,058 | https://pt.wikipedia.org/wiki/Wikip%C3%A9dia:P%C3%A1gina_principa
[0065/3000] 437 novos | total 8,487 | https://pt.wikipedia.org/wiki/Renascimento
[0066/3000]   0 novos | total 8,487 | https://pt.wikipedia.org/wiki/Física
[0067/3000]  32 novos | total 8,519 | https://pt.wikipedia.org/wiki/Wikip%C3%A9dia:Boas-vindas


...

[2958/3000] 339 novos | total 121,070 | https://pt.wikipedia.org/wiki/Europa


[2959/3000] 184 novos | total 121,254 | https://pt.wikipedia.org/wiki/Atmosfera_da_Terra


[2960/3000]  14 novos | total 121,268 | https://pt.wikipedia.org/wiki/Região_administrativa_especial
[2961/3000] 330 novos | total 121,591 | https://pt.wikipedia.org/wiki/Átomo
[2962/3000]   0 novos | total 121,591 | https://pt.wikipedia.org/wiki/Wikipédia:Sobre_a_Wikipédia
[2963/3000]   0 novos | total 121,591 | https://pt.wikipedia.org/wiki/Wikipédia:Portal_comunitário
[2964/3000]  25 novos | total 121,616 | https://agenciabrasil.ebc.com.br/economia/noticia/2026-06/jogos-d
[2965/3000]  27 novos | total 121,643 | https://agenciabrasil.ebc.com.br/economia/noticia/2025-11/apenas-
[2966/3000]  10 novos | total 121,653 | https://agenciabrasil.ebc.com.br/direitos-humanos/noticia/2026-05
[2967/3000]  12 novos | total 121,665 | https://agenciabrasil.ebc.com.br/internacional/noticia/2026-06/ar
[2968/3000]  11 novos | total 121,676 | https://agenciabrasil.ebc.com.br/tags/bernardinho
[2969/3000]  19 novos | total 121,695 | https://agenciabrasil.ebc.com.br/esportes/noticia/2026-06/em-hora
[2970/

[2980/3000]   0 novos | total 121,921 | https://vestibular.brasilescola.uol.com.br/imprimir/36062


[2981/3000]   0 novos | total 121,921 | https://vestibular.brasilescola.uol.com.br/imprimir/20191
[2982/3000]  29 novos | total 121,950 | https://vestibular.brasilescola.uol.com.br/enem/lista-espera-sisu


## 5. Lista todos os datasets disponíveis

In [ ]:
import os, glob

REPO = '/content/IA'
DATA_DIR = '/content/datasets'

# Datasets do repo (pequenos, já commitados)
repo_datasets = [
    f'{REPO}/dataset_portugues_br.txt',
    f'{REPO}/frases_treinamento.txt',
    f'{REPO}/dataset_v2.txt',
    f'{REPO}/dataset_v3.txt',
    f'{REPO}/dataset_wiki_pt.txt',
]

# Datasets baixados
downloaded = glob.glob(f'{DATA_DIR}/*.txt')

all_paths = [p for p in repo_datasets + downloaded if os.path.exists(p)]

print(f'Datasets encontrados ({len(all_paths)}):')
total_mb = 0
for p in all_paths:
    mb = os.path.getsize(p) / 1024 / 1024
    total_mb += mb
    print(f'  {os.path.basename(p):40s} {mb:7.1f} MB')
print(f'  {"TOTAL":40s} {total_mb:7.1f} MB')

## 6. Treina o modelo GemmaMicro

Parâmetros ajustáveis:
- `epochs` — mais épocas = mais tempo, melhor qualidade
- `ctx_len` — contexto. 512 usa mais VRAM.
- `batch_size` — reduza se der OOM (Out of Memory)

In [ ]:
import sys, os, glob
sys.path.insert(0, '/content/IA')
import nn

REPO      = '/content/IA'
DATA_DIR  = '/content/datasets'
DRIVE_DIR = '/content/drive/MyDrive/GemmaMicro'

# Reconstrói all_paths (caso célula 5 não tenha rodado)
repo_datasets = [
    f'{REPO}/dataset_portugues_br.txt',
    f'{REPO}/frases_treinamento.txt',
    f'{REPO}/dataset_v2.txt',
    f'{REPO}/dataset_v3.txt',
    f'{REPO}/dataset_wiki_pt.txt',
]
downloaded = glob.glob(f'{DATA_DIR}/*.txt')
all_paths = [p for p in repo_datasets + downloaded if os.path.exists(p)]

print(f'Datasets para treino ({len(all_paths)}):')
for p in all_paths:
    mb = os.path.getsize(p) / 1024 / 1024
    print(f'  {os.path.basename(p):40s} {mb:7.1f} MB')

# max_lines_per_file=500_000 evita OOM com wiki_pt.txt (2 GB)
# Wikipedia PT tem ~1M linhas — 500k já dá corpus rico
nn.train(
    data_paths=all_paths,
    save_dir=DRIVE_DIR,
    epochs=20,
    batch_size=64,              # reduza para 32 se der OOM
    lr=6e-4,
    ctx_len=512,
    accum_steps=4,              # batch efetivo = 64*4 = 256
    max_lines_per_file=500_000, # cap por arquivo — protege contra OOM
)


## 7. Testa inferência

In [ ]:
import torch, sys
sys.path.insert(0, '/content/IA')
from nn import GemmaMicro, BPETokenizer

DRIVE_DIR = '/content/drive/MyDrive/GemmaMicro'

tokenizer = BPETokenizer.load(f'{DRIVE_DIR}/tokenizer.json')
ckpt = torch.load(f'{DRIVE_DIR}/model.pt', map_location='cpu', weights_only=True)
model = GemmaMicro(vocab_size=tokenizer.vocab_size)
model.load_state_dict(ckpt)
model.eval()
print(f'Modelo: {sum(p.numel() for p in model.parameters()):,} parâmetros')

def gerar(prompt, max_tokens=120, temp=0.8, top_k=50):
    ids = tokenizer.encode(prompt, add_special=False)
    ids = torch.tensor([ids])
    with torch.no_grad():
        for _ in range(max_tokens):
            logits = model(ids[:, -512:])
            logit = logits[0, -1, :] / temp
            v, _ = torch.topk(logit, top_k)
            logit[logit < v[-1]] = float('-inf')
            next_id = torch.multinomial(torch.softmax(logit, -1), 1)
            ids = torch.cat([ids, next_id.unsqueeze(0)], dim=1)
            if next_id.item() == 258:  # <eos>
                break
    return tokenizer.decode(ids[0].tolist(), skip_special=False)

prompts = [
    'a capital do brasil é',
    'a inteligência artificial',
    'o brasil é um país',
    'a língua portuguesa',
]
for p in prompts:
    print(f'\n→ {p}')
    print(gerar(p))

## 8. (Opcional) Baixa modelo do Drive para /content

In [ ]:
import shutil, os

DRIVE_DIR = '/content/drive/MyDrive/GemmaMicro'
LOCAL_DIR = '/content/gemma_micro'
os.makedirs(LOCAL_DIR, exist_ok=True)

for f in ['model.pt', 'tokenizer.json']:
    src = f'{DRIVE_DIR}/{f}'
    dst = f'{LOCAL_DIR}/{f}'
    if os.path.exists(src):
        shutil.copy(src, dst)
        mb = os.path.getsize(dst)/1024/1024
        print(f'Copiado {f}: {mb:.1f} MB')

print('Pronto. Use load_model(\'gemma_micro\') localmente.')